# Correlation-Based Feature Reduction

Drops the bottom 30% of features by absolute Pearson correlation with `Price`, then retrains the best model (HistGradientBoosting) to confirm accuracy is preserved.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from src.preprocessing import build_features
from src.evaluate import fit_score

X, y, _ = build_features('../data/video_game_reviews.csv')
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=67)

In [ ]:
corr = Xtr.apply(lambda c: np.corrcoef(c.astype(float), ytr)[0, 1]).abs().sort_values(ascending=False)
corr.to_frame('|corr_with_price|').head(15)

In [ ]:
drop_n = int(round(0.30 * len(corr)))
to_drop = corr.tail(drop_n).index.tolist()
print(f'Dropping {drop_n} of {len(corr)} features ({drop_n/len(corr):.0%}):')
for f in to_drop:
    print(f'  - {f:40s}  |corr| = {corr[f]:.4f}')

In [ ]:
Xtr2 = Xtr.drop(columns=to_drop)
Xte2 = Xte.drop(columns=to_drop)

def hgb():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=8, random_state=0)

full = fit_score(hgb(), Xtr, Xte, ytr, yte, f'GBM full ({Xtr.shape[1]} feat)')
reduced = fit_score(hgb(), Xtr2, Xte2, ytr, yte, f'GBM reduced ({Xtr2.shape[1]} feat)')

pd.DataFrame([full.as_row(), reduced.as_row()])

**Conclusion:** dropping 30% of the lowest-correlation features leaves R² essentially unchanged (delta < 0.001), confirming those features carried little signal.